In [1]:
import torch
import numpy as np
from src.datasets.processed_dataset import PreprocessedGraphDataset
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt

from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel,  OPTForCausalLM

from src.model.stage_models import *
from src.model.utils import *
from src.loss.molcaloss import MolCALoss
from src.datasets.collate import *

/Users/sgerasimov/Desktop/altegrad/MolCA/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = 'cpu'

In [3]:
bert_name = "allenai/scibert_scivocab_uncased"

tokenizer = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")

In [4]:
dataset = PreprocessedGraphDataset(graph_path="./data/train_graphs.pkl")

Loading graphs from: ./data/train_graphs.pkl
Loaded 31008 graphs


In [5]:
loader = DataLoader(dataset, batch_size=64, shuffle=True, collate_fn=TrainCollater(tokenizer, 128))

In [7]:
for graphs, text_ids, attention_mask in loader:
    break

# Stage 1

In [8]:
def get_scheduler(optimizer, max_steps, warmup_steps, min_lr):
    warmup_scheduler = LinearLR(
        optimizer, 
        start_factor=0.01, 
        total_iters=warmup_steps
    )
    
    cosine_scheduler = CosineAnnealingLR(
        optimizer, 
        T_max=(max_steps - warmup_steps), 
        eta_min=min_lr
    )
    
    scheduler = SequentialLR(
        optimizer, 
        schedulers=[warmup_scheduler, cosine_scheduler], 
        milestones=[warmup_steps]
    )
    
    return scheduler

In [9]:
warmup_steps = 1000
init_lr = 1e-4
min_lr = 1e-5
weight_decay = 0.05
warmup_lr = 1e-6
retrieval_eval_epoch = 10
num_epochs = 50
max_steps = len(loader)*num_epochs

In [10]:
stage1model = Stage1Wrapper(gnn_pretrained="./checkpoints/graphcl_80.pth")

In [11]:
optimizer = torch.optim.AdamW(stage1model.parameters(), lr=init_lr, weight_decay=weight_decay)
scheduler = get_scheduler(optimizer, max_steps, warmup_steps, min_lr)
criterion = MolCALoss(learnable_temp=False)

In [13]:
total_loss = []
total_loss_itc = []
total_loss_itm = []
total_loss_lm = []

for epoch in tqdm(range(1, num_epochs+1), desc="Epoch"):
    epoch_loss = []
    epoch_loss_itc = []
    epoch_loss_itm = []
    epoch_loss_lm = []
    for graphs, text_ids, attention_mask in loader:
        graph_feats, t_feats = stage1model(graphs, text_ids, attention_mask)
        loss = criterion(graph_feats, t_feats, stage1model.itm_head)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        epoch_loss.append(loss.detach().cpu().numpy())

    total_loss.append(np.mean(epoch_loss))

    plt.figure(figsize=(12, 5))

    plt.tight_layout()
    plt.show()
    
    if epoch%retrieval_eval_epoch==0:
        pass
# Опционально: сохранение графиков в файл
# plt.savefig(f"all_checkpoints/{args.filename}/loss_plots.png")

    

Epoch:   0%|          | 0/50 [00:11<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
torch.save(stage1model.adapter.state_dict(), "mlp_adapter_stage1.pth")

# Stage 2

In [10]:
model_name = "facebook/galactica-1.3b"

galactica_tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. Добавляем специальный токен для молекулы, если его нет в словаре
special_tokens = {"additional_special_tokens": ["<mol>"]}
galactica_tokenizer.add_special_tokens(special_tokens)
galactica_tokenizer.pad_token = "<pad>"
galactica_tokenizer.padding_side = "left"

if galactica_tokenizer.eos_token is None:
    galactica_tokenizer.eos_token = "</s>"
    galactica_tokenizer.eos_token_id = 2

In [11]:
loader = DataLoader(dataset, batch_size=2, collate_fn=TrainCollater2(galactica_tokenizer, None))

In [12]:
batch_graph, input_ids, attention_mask, labels, mol_mask = next(iter(loader))

In [ ]:
stage2model = Stage2Wrapper(gnn_pretrained='checkpoints/graphcl_80.pth').to(device)

# 3. Загрузка модели
# device_map="auto" сама распределит модель, 
# torch_dtype=torch.float16 критически важен для экономии памяти на P100
model = OPTForCausalLM.from_pretrained(
    model_name, 
    dtype=torch.float32, 
    device_map=device
)

# 4. Важно: если мы добавили токен в токенизатор, 
# нужно расширить матрицу эмбеддингов модели
model.resize_token_embeddings(len(galactica_tokenizer))
freeze_model(model)

No such file


In [16]:
warmup_steps = 1000
init_lr = 1e-4
min_lr = 1e-5
weight_decay = 0.05
warmup_lr = 1e-6
retrieval_eval_epoch = 10
num_epochs = 10
max_steps = len(loader)*num_epochs

In [ ]:
optimizer = torch.optim.AdamW(stage2model.parameters(), lr=init_lr, weight_decay=weight_decay)
scheduler = get_scheduler(optimizer, max_steps, warmup_steps, min_lr)

In [ ]:
total_loss = []


for epoch in tqdm(range(1, num_epochs+1), desc="Epoch"):
    epoch_loss = []
    for batch_graph, input_ids, attention_mask, labels, mol_mask in loader:
        
        batch_graph, input_ids, attention_mask =  batch_graph.to(device), input_ids.to(device), attention_mask.to(device),
        labels, mol_mask = labels.to(device), mol_mask.to(device)
        
        graph_embs= stage2model(batch_graph)
        
        embs = model.get_input_embeddings()(input_ids)

        embs[mol_mask] = graph_embs.reshape(-1, graph_embs.shape[-1]).to(embs.dtype)

        assert embs.shape[1] == attention_mask.shape[1] == labels.shape[1]

        outputs = model(
            inputs_embeds=embs,       
            attention_mask=attention_mask,
            labels=labels,           
            return_dict=True
        )

        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        epoch_loss.append(loss.detach().cpu().numpy())
    

    total_loss.append(np.mean(epoch_loss))

    plt.figure(figsize=(12, 5))

    plt.tight_layout()
    plt.show()

    if epoch%retrieval_eval_epoch==0:
        pass

# Опционально: сохранение графиков в файл
# plt.savefig(f"all_checkpoints/{args.filename}/loss_plots.png")

    

In [ ]:
torch.tensor([1.2]).dtype

AttributeError: 'Tensor' object has no attribute 'dtypes'

In [ ]:
torch.save(stage2model.gnn.state_dict(), "checkpoints/gnn_stage2.pth")
torch.save(stage2model.adapter.state_dict(), "checkpoints/mlp_adapter_stage2.pth")

# Stage 3

In [80]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM

# 1. Загружаем базовую модель Galactica
model_name = "facebook/galactica-1.3b" # или другая версия
model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    device_map="cpu",
    trust_remote_code=True
)
lora_config = {
"base_model_name_or_path": None,
"bias": "none",
"fan_in_fan_out": False,
"inference_mode": False,
"init_lora_weights": True,
"lora_alpha": 32,
"lora_dropout": 0.1,
"target_modules": ["q_proj", "v_proj", "out_proj", "fc1", "fc2"],
"peft_type": "LORA",
"r": 16,
"modules_to_save": None,
"task_type": "CAUSAL_LM"
}

# 2. Настройка LoRA
lora_config = LoraConfig(
    **lora_config
)

# 3. Оборачиваем модель
model = get_peft_model(model, lora_config)

# Выведем количество обучаемых параметров
model.print_trainable_parameters()

trainable params: 12,582,912 || all params: 1,327,783,936 || trainable%: 0.9477


In [ ]:
stage3model = Stage2Wrapper(gnn_pretrained="checkpoints/gnn_stage2.pth", adapter_pretrained="checkpoints/mlp_adapter_stage2.pth")

No such file
No such file


In [90]:
import torch.optim as optim

warmup_steps = 1000
init_lr = 1e-4
min_lr = 1e-5
weight_decay = 0.05
warmup_lr = 1e-6
retrieval_eval_epoch = 10
num_epochs = 10
max_steps = len(loader)*num_epochs

# Настраиваем AdamW (стандарт для Stage 3)
optimizer = optim.AdamW([
    {
        # LoRA слои Galactica (уже имеющаяся база знаний)
        "params": [p for p in model.parameters() if p.requires_grad],
        "lr": 5e-5, 
        "weight_decay": weight_decay
    },
    {
        # Твой MLP-адаптер (новый мост, учится с нуля)
        "params": [p for p in stage3model.parameters() if p.requires_grad],
        "lr": 1e-4, # Даем чуть больше свободы
        "weight_decay": weight_decay
    }
])

In [95]:
from torchmetrics.text import BleuScore, RougeLScore

ImportError: cannot import name 'BleuScore' from 'torchmetrics.text' (/Users/sgerasimov/Desktop/altegrad/MolCA/.venv/lib/python3.11/site-packages/torchmetrics/text/__init__.py)

In [97]:
from sacrebleu import corpus_bleu
from bert_score import score as bertscore


In [106]:
# Гипотезы (что выдала Galactica)
sys = ["the molecule is a derivative of adenosine", "it is a human metabolite"]

# Референсы (эталоны из датасета) — важно: список списков!
refs = [
    ["the molecule is of adenosine"], 
    ["i"]
]

bleu = corpus_bleu(sys, refs)
print(bleu.score)

30.739407647563215


In [ ]:
predictions = [
    "The molecule is a derivative of adenosine with a phosphate group.",
    "This compound acts as a human metabolite in the liver."
]

references = [
    "It is an adenosine derivative containing a phosphorus atom.",
    "The substance is involved in human metabolism."
]
_, _, F1 = bertscore(
        predictions, 
        references, 
        lang="en", 
        device="cpu", # Обязательно считаем на GPU (A100 это сделает мгновенно)
        verbose=False
    )
F1.mean().item()

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


0.9353856444358826

In [107]:
bertscore


<function bert_score.score.score(cands, refs, model_type=None, num_layers=None, verbose=False, idf=False, device=None, batch_size=64, nthreads=4, all_layers=False, lang=None, return_hash=False, rescale_with_baseline=False, baseline_path=None, use_fast_tokenizer=False)>